# パイプライン用ノートブック: 検知 → 相関(±10分) → AI修正案 → incidents書き込み → exit value返却

**役割**: このノートブックは検知ロジックのみ。メール送信はパイプラインの Office 365 Outlook アクティビティが行います。

**エラー判定**: silver_app は `level` 列、silver_oracle は `ora_code` 列（ORAコードが入っている行をエラーとみなす）。

**戻り値**: `notebookutils.notebook.exit()` で JSON文字列 `{"incident_count", "subject", "body"}` を返し、パイプラインの If 条件 / Outlook 本文から参照します。

**前提**: Lakehouse `loghandson` を既定Lakehouseとしてアタッチ。

In [ ]:
# 依存関係のインストール(最初のセル・単独で実行)
%pip install -q openai

In [ ]:
# 設定(自分の環境に合わせて埋める)
APP_TABLE      = "silver_app"
ORACLE_TABLE   = "silver_oracle"
INCIDENT_TABLE = "incidents"      # 出力先(初回実行時に自動作成)
TS_COL         = "event_time"     # 両テーブル共通のタイムスタンプ列
MSG_COL        = "message"        # 両テーブル共通の本文列

LEVEL_COL      = "level"          # silver_app のエラー判定列(== 'ERROR')
ORA_CODE_COL   = "ora_code"       # silver_oracle のエラー判定列(ORAコードが入る)

WINDOW_MIN     = 10

AI_MODEL    = "gpt-5-mini"        # 代替: "gpt-5.1" / "gpt-4.1"
API_VERSION = "2025-04-01-preview"
# メール設定はここには無い(パイプラインのOutlookアクティビティで設定する)

In [ ]:
# (1) silver_app から最新のERROR 1件(level == 'ERROR' で判定)
from pyspark.sql import functions as F
from datetime import timedelta
import json

app_df = (spark.table(APP_TABLE)
            .where(F.upper(F.col(LEVEL_COL)) == "ERROR")
            .orderBy(F.col(TS_COL).desc())
            .limit(1))
app_err = app_df.first()
if app_err is None:
    print("ERRORなし。終了。")
    notebookutils.notebook.exit(json.dumps({"incident_count": 0}))

app_ts  = app_err[TS_COL]
app_msg = app_err[MSG_COL]
print(f"[App ERROR] {app_ts}\n{app_msg}\n")

In [ ]:
# 重複送信の防止: 既に処理済みの最新エラーなら終了(同じエラーを毎回送らない)
already = None
try:
    already = spark.table(INCIDENT_TABLE).agg(F.max("app_time")).first()[0]
except Exception:
    already = None  # 初回はテーブル未作成
if already is not None and app_ts <= already:
    print("最新エラーは処理済み。新規なし。終了。")
    notebookutils.notebook.exit(json.dumps({"incident_count": 0}))

In [ ]:
# (2) silver_oracle から ±WINDOW_MIN分のエラー(ora_code が入っている行で判定)
low  = app_ts - timedelta(minutes=WINDOW_MIN)
high = app_ts + timedelta(minutes=WINDOW_MIN)
ora_df = (spark.table(ORACLE_TABLE)
            .where(F.col(ORA_CODE_COL).isNotNull()
                   & (F.trim(F.col(ORA_CODE_COL).cast("string")) != ""))
            .where(F.col(TS_COL).between(F.lit(low), F.lit(high)))
            .orderBy(F.col(TS_COL).asc()))
ora_rows = ora_df.collect()
print(f"[Oracle ERROR ±{WINDOW_MIN}min] {len(ora_rows)} 件")
ora_block = ("\n".join(f"- {r[TS_COL]}: [{r[ORA_CODE_COL]}] {r[MSG_COL]}" for r in ora_rows)
             if ora_rows else "(同時間帯に Oracle のエラーはありませんでした)")

In [ ]:
# (3) AIモデルで修正案を生成
from synapse.ml.fabric.credentials import get_openai_httpx_sync_client
import openai
client = openai.AzureOpenAI(http_client=get_openai_httpx_sync_client(), api_version=API_VERSION)

prompt = f"""あなたはシステム運用のエキスパートです。
以下のアプリケーションエラーと、その前後{WINDOW_MIN}分以内に発生したOracleのエラー(ORAコード付き)を踏まえ、
(1)推定される原因 (2)具体的な修正手順 を日本語で簡潔に提案してください。

# アプリケーションエラー(発生時刻 {app_ts})
{app_msg}

# 同時間帯のOracleエラー
{ora_block}
"""
resp = client.chat.completions.create(
    model=AI_MODEL,
    messages=[{"role":"system","content":"簡潔で実践的な障害対応の提案を返す。"},
              {"role":"user","content":prompt}],
)
suggestion = resp.choices[0].message.content
print("[AI 修正案]\n" + suggestion)

In [ ]:
# 本文を組み立てて incidents テーブルへ追記
from pyspark.sql import Row
import datetime

ora_html = ("<br>".join(f"・{r[TS_COL]}: [{r[ORA_CODE_COL]}] {r[MSG_COL]}" for r in ora_rows)
            if ora_rows else "(同時間帯に Oracle のエラーはありませんでした)")
html_body = f"""<h3>エラー検知通知</h3>
<p><b>アプリケーションエラー</b>(発生時刻 {app_ts})<br>{app_msg}</p>
<p><b>同時間帯のOracleエラー(±{WINDOW_MIN}分)</b><br>{ora_html}</p>
<hr>
<p><b>AIによる修正案</b><br>{suggestion.replace(chr(10), '<br>')}</p>"""
subject = f"[エラー検知] {app_ts} アプリエラー + Oracle {len(ora_rows)}件"

inc = spark.createDataFrame([Row(
    detected_at = datetime.datetime.now(),
    app_time    = app_ts,
    app_message = app_msg,
    oracle_count= len(ora_rows),
    oracle_block= ora_block,
    suggestion  = suggestion,
    subject     = subject,
    body_html   = html_body,
)])
inc.write.mode("append").saveAsTable(INCIDENT_TABLE)
print("incidents に追記しました。")

In [ ]:
# exit value を返す(パイプラインの If / Outlook から参照)
notebookutils.notebook.exit(json.dumps({
    "incident_count": 1,          # ここまで来た=新規インシデントあり
    "subject": subject,
    "body": html_body,
}, ensure_ascii=False))